# KYC: Graph-Native Identity Resolution

**Run on dedicated compute. Runs after the three-step enrichment pipeline (`03_neo4j_ingest` → `04_gds_enrichment` → `05_pull_gold_tables`).**

Money movement found the fraud ring. This notebook adds the two things a warehouse cannot reach on its own: *identity resolution* proves the ring is one person wearing eight masks, and a thin *knowledge layer* names the policy, definition, and data sources that made the call.

It runs against the KYC identity layer loaded in `03_neo4j_ingest` (`:Customer` / `:Phone` / `:Address` joined by `OWNS` / `HAS_PHONE` / `HAS_ADDRESS`) and the Louvain `community_id` written in `04_gds_enrichment`. Three parts compute the graph features and write them back to Delta. The live presenter walkthrough is the companion notebook `07_kyc_walkthrough`.

## The planted story ring

Eight accounts sit inside fraud ring 0, owned by eight customers who share a small set of identifiers:

| Identifier | Value | Customers (by account_id) |
|-----------|-------|---------------------------|
| Phone A | 312-555-0142 | 368, 927, 1033, 1696 |
| Phone B | 312-555-0143 | 2184, 2216, 2612, 3003 |
| Address | 1247 W Cermak Rd, Chicago, IL 60608 | 1033, 1696, 2184, 2216 |

The address is the bridge: it spans two customers from Phone A (1033, 1696) and two from Phone B (2184, 2216), so all eight collapse into one Weakly Connected Component even though no single phone connects them all. That is the traversal a warehouse cannot express in one hop.

In [ ]:
%pip install graphdatascience --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Configuration and Connect

Run after the `%pip install` kernel restart above. Same catalog, schema, and Neo4j secrets as the rest of the workshop.

In [ ]:
CATALOG   = "graph-on-databricks"
SCHEMA    = "graph-enriched-schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

NEO4J_OPTS = {
    "url":                    NEO4J_URI,
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
}

# Split the Neo4j read into concurrent tasks; label mode partitions internally
# by id range so the output is unchanged.
NEO4J_READ_PARTITIONS = "8"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

In [ ]:
from graphdatascience import GraphDataScience
from pyspark.sql import functions as F

gds = GraphDataScience(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print(f"Connected  |  GDS version: {gds.version()}")

## 1. Sanity Check: Is the Identity Layer Loaded?

The identity resolution below needs the `:Customer` / `:Phone` / `:Address` nodes from `03_neo4j_ingest`. If the customer count is zero, re-run the identity-layer section (section 8) of `03_neo4j_ingest` before continuing.

In [ ]:
counts = gds.run_cypher("""
    RETURN count { (a:Account) }  AS accounts,
           count { (c:Customer) } AS customers,
           count { (p:Phone) }    AS phones,
           count { (ad:Address) } AS addresses
""")
print(counts.to_string(index=False))

assert int(counts["customers"].iloc[0]) > 0, (
    "No :Customer nodes found — run the KYC identity layer (section 8) of "
    "03_neo4j_ingest before this notebook."
)

---
## Part A: Identity Resolution with Weakly Connected Components

WCC picks up one customer and sees everyone who lifts with them through any chain of shared identifiers. A normal customer is an island of one. The story ring is one island, tied together by two phones and one shared address.

### Step 1: Project the `customer_identity` graph (UNDIRECTED)

Project only the identity layer: `:Customer` / `:Phone` / `:Address` nodes and the `HAS_PHONE` / `HAS_ADDRESS` edges, undirected so a shared identifier connects its owners in both directions.

In [ ]:
gds.run_cypher("CALL gds.graph.drop('customer_identity', false) YIELD graphName")

G, stats = gds.graph.project(
    "customer_identity",
    ["Customer", "Phone", "Address"],
    {
        "HAS_PHONE":   {"orientation": "UNDIRECTED"},
        "HAS_ADDRESS": {"orientation": "UNDIRECTED"},
    },
)
print(
    f"projected '{G.name()}': {stats['nodeCount']:,} nodes, "
    f"{stats['relationshipCount']:,} relationships"
)

### Step 2: WCC → `identity_cluster_id`

Every node in a connected component gets the same `identity_cluster_id`. Customers linked by any chain of shared phones or addresses land in one component.

In [ ]:
wcc = gds.wcc.write(G, writeProperty="identity_cluster_id")
print(
    f"componentCount={int(wcc['componentCount']):,}  "
    f"propertiesWritten={int(wcc['nodePropertiesWritten']):,}"
)
gds.graph.drop(G)

### Step 3: `identity_cluster_size` per customer

Cluster size counts customers only. WCC also labels the `:Phone` and `:Address` nodes in each component, but those are identifiers, not members. Expect `customers_in_shared_clusters` = 8 — the eight story customers.

In [ ]:
sized = gds.run_cypher("""
    MATCH (c:Customer)
    WITH c.identity_cluster_id AS cid, collect(c) AS members
    UNWIND members AS c
    SET c.identity_cluster_size = size(members)
    WITH cid, size(members) AS cluster_size
    RETURN count(DISTINCT cid) AS clusters,
           sum(CASE WHEN cluster_size > 1 THEN 1 ELSE 0 END)
               AS customers_in_shared_clusters
""")
print(sized.to_string(index=False))

### Step 4: `shared_phone_count` / `shared_address_count` per customer

For each customer, count the distinct *other* customers reached through a shared phone, and through a shared address. Expect 8 customers sharing a phone (four per phone group) and 4 sharing the address.

In [ ]:
for rel, prop in (
    ("HAS_PHONE", "shared_phone_count"),
    ("HAS_ADDRESS", "shared_address_count"),
):
    row = gds.run_cypher(f"""
        MATCH (c:Customer)
        OPTIONAL MATCH (c)-[:{rel}]->()<-[:{rel}]-(other:Customer)
        WITH c, count(DISTINCT other) AS n
        SET c.{prop} = n
        RETURN sum(CASE WHEN n > 0 THEN 1 ELSE 0 END) AS sharing
    """)
    print(f"{prop}: {int(row['sharing'].iloc[0]):,} customers share")

### Step 5: Propagate identity properties to `:Account` via `OWNS`

The Genie-facing columns live on `:Account`, so copy the four identity properties from each customer to the account they own. `05_pull_gold_tables` reads them from `:Account` nodes.

In [ ]:
propagated = gds.run_cypher("""
    MATCH (c:Customer)-[:OWNS]->(a:Account)
    SET a.identity_cluster_id = c.identity_cluster_id,
        a.identity_cluster_size = c.identity_cluster_size,
        a.shared_phone_count = c.shared_phone_count,
        a.shared_address_count = c.shared_address_count
    RETURN count(a) AS accounts_updated
""")
print(f"accounts_updated={int(propagated['accounts_updated'].iloc[0]):,}")

---
## Part B: Knowledge Layer and Classification

The customer's question is not only *who* violates the policy. It is *which business definition* and *which data source* made the call. That explanation is a traversal too.

### Step 6: Build the knowledge layer

A thin semantic and provenance layer — `:Policy` / `:BusinessTerm` / `:BusinessRule` / `:DataSource` — so "which policy, definition, and data sources flagged this customer" is a traversal, not tribal knowledge. All `MERGE`, so it is safe to re-run.

In [ ]:
KYC_BUSINESS_TERM = "Shared Identity Ring"

gds.run_cypher(
    """
    MERGE (p:Policy {policy_id: 'KYC-CIP-001'})
      SET p.name = 'Customer Identification Program (KYC)',
          p.authority = 'FinCEN 31 CFR 1020.220',
          p.description = 'Requires verifying customer identity and detecting customers operating under shared or synthetic identities.'
    MERGE (term:BusinessTerm {name: $term})
      SET term.description = 'A group of customers linked into one identity cluster by shared phone numbers or addresses, indicating possible synthetic-identity or structuring activity.'
    MERGE (rule:BusinessRule {rule_id: 'KYC-WCC-001'})
      SET rule.name = 'Shared-identity WCC cluster',
          rule.logic = 'Weakly Connected Components over (:Customer)-[:HAS_PHONE|HAS_ADDRESS]->() ; flag every customer whose identity_cluster_size > 1.',
          rule.threshold = 1
    MERGE (phone:DataSource {name: 'silver.customers.phone'})
      SET phone.description = 'Customer phone column; feeds the :Phone identity nodes via HAS_PHONE.'
    MERGE (addr:DataSource {name: 'silver.customers.address'})
      SET addr.description = 'Customer address column; feeds the :Address identity nodes via HAS_ADDRESS.'
    MERGE (term)-[:GOVERNED_BY]->(p)
    MERGE (term)-[:DEFINED_BY]->(rule)
    MERGE (rule)-[:DERIVED_FROM]->(phone)
    MERGE (rule)-[:DERIVED_FROM]->(addr)
    """,
    params={"term": KYC_BUSINESS_TERM},
)
print(
    "knowledge layer ready: Policy, BusinessTerm, BusinessRule, "
    "2 DataSource + provenance edges"
)

### Step 7: Classify shared-identity customers → `:CLASSIFIED_AS`

Every customer whose WCC cluster holds more than one customer is a member of a shared-identity ring. The `:CLASSIFIED_AS` edge carries a human-readable reason and the cluster it was derived from, so the classification explains itself. The first query clears any stale edges from a prior run before the reclassification. Expect 8 customers classified.

In [ ]:
cleared = gds.run_cypher("""
    MATCH (:Customer)-[r:CLASSIFIED_AS]->(:BusinessTerm)
    DELETE r RETURN count(r) AS deleted
""")
print(f"deleted {int(cleared['deleted'].iloc[0]):,} stale classification edges")

classified = gds.run_cypher(
    """
    MATCH (term:BusinessTerm {name: $term})
    MATCH (c:Customer) WHERE c.identity_cluster_size > 1
    MERGE (c)-[r:CLASSIFIED_AS]->(term)
    SET r.reason = 'shares ' + toString(c.shared_phone_count) +
                   ' phone(s) and ' + toString(c.shared_address_count) +
                   ' address with ' + toString(c.identity_cluster_size - 1) +
                   ' other customer(s) in identity cluster ' +
                   toString(c.identity_cluster_id),
        r.evaluatedAt = datetime(),
        r.cluster_id = c.identity_cluster_id,
        r.cluster_size = c.identity_cluster_size
    RETURN count(r) AS classified
    """,
    params={"term": KYC_BUSINESS_TERM},
)
print(f"customers classified as '{KYC_BUSINESS_TERM}': {int(classified['classified'].iloc[0]):,}")

---
## Part C: Write the KYC Columns Back to `gold_accounts`

Nothing in the graph reaches a Genie query except what lands in Delta. `05_pull_gold_tables` built `gold_accounts`; this augments it in place with the four graph-derived KYC columns so they sit beside `risk_score` and `community_id`. Their Unity Catalog comments tell Genie that any value above the baseline signals synthetic-identity risk.

This is the Multi-Hop Native pattern closing its loop: data started in Databricks, multi-hop detection ran in the graph, results return to Delta.

In [ ]:
GOLD_ACCOUNTS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.gold_accounts"

# Read the four KYC properties back from the enriched :Account nodes.
kyc_df = (
    spark.read
    .format("org.neo4j.spark.DataSource")
    .options(**NEO4J_OPTS)
    .option("labels", "Account")
    .option("partitions", NEO4J_READ_PARTITIONS)
    .load()
    .select(
        F.col("account_id").cast("long"),
        F.col("shared_phone_count").cast("long"),
        F.col("shared_address_count").cast("long"),
        F.col("identity_cluster_id").cast("long"),
        F.col("identity_cluster_size").cast("long"),
    )
)
print(f"Read {kyc_df.count():,} Account nodes with KYC properties")

# Add the four columns to gold_accounts if they are not already present, each
# with the Unity Catalog comment Genie reads. No apostrophes in the comments —
# they are embedded in single-quoted SQL string literals.
KYC_COLUMNS = [
    ("shared_phone_count", "BIGINT",
     "Number of other customers who share a phone number with the owner of this account. 0 is the clean baseline; any value above 0 is a synthetic-identity risk signal."),
    ("shared_address_count", "BIGINT",
     "Number of other customers who share a mailing address with the owner of this account. 0 is the clean baseline; any value above 0 is a synthetic-identity risk signal."),
    ("identity_cluster_id", "BIGINT",
     "Weakly Connected Component id over the shared phone and address graph. Accounts whose owners are linked by any chain of shared identifiers share this id."),
    ("identity_cluster_size", "BIGINT",
     "Number of customers in the identity cluster for this account. 1 means a unique identity; greater than 1 means the owner shares identifiers with other customers (a shared-identity ring)."),
]

existing = {f.name for f in spark.table(GOLD_ACCOUNTS_TABLE).schema.fields}
missing = [(n, t, c) for n, t, c in KYC_COLUMNS if n not in existing]
if missing:
    cols_ddl = ", ".join(f"{n} {t} COMMENT '{c}'" for n, t, c in missing)
    spark.sql(f"ALTER TABLE {GOLD_ACCOUNTS_TABLE} ADD COLUMNS ({cols_ddl})")
    print(f"Added {len(missing)} KYC columns to gold_accounts")
else:
    print("KYC columns already present on gold_accounts")

# MERGE the values in by account_id — every gold account has a matching row.
kyc_df.createOrReplaceTempView("kyc_updates")
spark.sql(f"""
    MERGE INTO {GOLD_ACCOUNTS_TABLE} AS t
    USING kyc_updates AS s
    ON t.account_id = s.account_id
    WHEN MATCHED THEN UPDATE SET
        t.shared_phone_count    = s.shared_phone_count,
        t.shared_address_count  = s.shared_address_count,
        t.identity_cluster_id   = s.identity_cluster_id,
        t.identity_cluster_size = s.identity_cluster_size
""")

display(
    spark.table(GOLD_ACCOUNTS_TABLE)
    .filter("shared_phone_count > 0 OR shared_address_count > 0")
    .select(
        "account_id", "account_name", "community_id",
        "identity_cluster_id", "identity_cluster_size",
        "shared_phone_count", "shared_address_count",
    )
    .orderBy("identity_cluster_id", "account_id")
)